In [1]:
from langchain_openai import ChatOpenAI

In [3]:
llm = ChatOpenAI(
    temperature = 0.0,
    model = 'gpt-5.4-mini'
)
resp = llm.invoke("The first female doctor in the world is ....")
print(resp.content)

The first recorded female doctor in history is **Merit-Ptah**, an ancient Egyptian physician, often cited as having lived around **2700 BCE**.

If you mean the **first modern female doctor** or the **first woman to receive a medical degree**, that is usually **Elizabeth Blackwell**, who became the first woman to receive a medical degree in the U.S. in **1849**.


In [5]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://jobs.nike.com/job/R-33460")
pg_data = loader.load().pop().page_content
print(pg_data)





















Nike Careers











































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu



Select a Lang

In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """
### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE): 
"""
)

chains = prompt | llm
resp = chains.invoke({'page_data':pg_data})
print(resp.content)

[
  {
    "role": "Kaufmann/-frau im Einzelhandel (Ausbildung) - 37.5H - Metzingen (w/m/d)",
    "experience": "Not specified",
    "skills": ["Retail", "Customer service", "Sales", "Training"],
    "description": "Retail store role in Metzingen, Baden-Wurttemberg, Germany."
  },
  {
    "role": "Part-time Supervisor - Nike Legacy Place",
    "experience": "Not specified",
    "skills": ["Retail leadership", "Customer service", "Team supervision", "Operations"],
    "description": "Part-time supervisory role at Nike Legacy Place in Dedham, Massachusetts, United States."
  },
  {
    "role": "Nike Operational Teamleiter (Lead) (w/m/d) – 38.5 (h) – NFS Parndorf",
    "experience": "Not specified",
    "skills": ["Retail operations", "Leadership", "Team management", "Store execution"],
    "description": "Operational team lead role at NFS Parndorf in Austria."
  },
  {
    "role": "Coach Stockroom I Gerente de Almacén - NFS MX Tijuana",
    "experience": "Not specified",
    "skills": ["S

In [8]:
type(resp.content)

str

In [9]:
from langchain_core.output_parsers import JsonOutputParser
json =JsonOutputParser()
json_res = json.parse(resp.content)
json_res

[{'role': 'Kaufmann/-frau im Einzelhandel (Ausbildung) - 37.5H - Metzingen (w/m/d)',
  'experience': 'Not specified',
  'skills': ['Retail', 'Customer service', 'Sales', 'Training'],
  'description': 'Retail store role in Metzingen, Baden-Wurttemberg, Germany.'},
 {'role': 'Part-time Supervisor - Nike Legacy Place',
  'experience': 'Not specified',
  'skills': ['Retail leadership',
   'Customer service',
   'Team supervision',
   'Operations'],
  'description': 'Part-time supervisory role at Nike Legacy Place in Dedham, Massachusetts, United States.'},
 {'role': 'Nike Operational Teamleiter (Lead) (w/m/d) – 38.5 (h) – NFS Parndorf',
  'experience': 'Not specified',
  'skills': ['Retail operations',
   'Leadership',
   'Team management',
   'Store execution'],
  'description': 'Operational team lead role at NFS Parndorf in Austria.'},
 {'role': 'Coach Stockroom I Gerente de Almacén - NFS MX Tijuana',
  'experience': 'Not specified',
  'skills': ['Stockroom management',
   'Inventory con

In [10]:
type(json_res)

list

In [11]:
import pandas as pd
df = pd.read_csv("my_portfolio.csv")
df.head()

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio


In [13]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name='portfolio')

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row['Techstack'],
                       metadatas={'links':row['Links']},
                       ids = [str(uuid.uuid4())])

C:\Users\Nayan\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [06:22<00:00, 218kiB/s] 


In [16]:
links = collection.query(query_texts=['Experience in Python', 'Expertise in React Native'],
                         n_results=2).get('metadatas',[])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/react-native-portfolio'},
  {'links': 'https://example.com/react-portfolio'}]]

In [17]:
job = json_res

In [20]:
job

[{'role': 'Kaufmann/-frau im Einzelhandel (Ausbildung) - 37.5H - Metzingen (w/m/d)',
  'experience': 'Not specified',
  'skills': ['Retail', 'Customer service', 'Sales', 'Training'],
  'description': 'Retail store role in Metzingen, Baden-Wurttemberg, Germany.'},
 {'role': 'Part-time Supervisor - Nike Legacy Place',
  'experience': 'Not specified',
  'skills': ['Retail leadership',
   'Customer service',
   'Team supervision',
   'Operations'],
  'description': 'Part-time supervisory role at Nike Legacy Place in Dedham, Massachusetts, United States.'},
 {'role': 'Nike Operational Teamleiter (Lead) (w/m/d) – 38.5 (h) – NFS Parndorf',
  'experience': 'Not specified',
  'skills': ['Retail operations',
   'Leadership',
   'Team management',
   'Store execution'],
  'description': 'Operational team lead role at NFS Parndorf in Austria.'},
 {'role': 'Coach Stockroom I Gerente de Almacén - NFS MX Tijuana',
  'experience': 'Not specified',
  'skills': ['Stockroom management',
   'Inventory con

In [21]:
prompt_emails = PromptTemplate.from_template(
    """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
)

chain_email = prompt_emails | llm
resp = chain_email.invoke({'job_description': str(job),
                           'link_list':links})
print(resp.content)

Subject: Helping Nike streamline retail operations, training, and customer experience

Hi Team,

I’m Mohan, a Business Development Executive at AtliQ. I came across your open roles across retail operations, store leadership, stockroom management, and customer-facing positions, and I wanted to reach out with a quick introduction.

AtliQ helps retail and consumer brands build AI and software solutions that improve store operations, workforce efficiency, and customer experience. For teams like yours, we can support with:

- Retail workflow automation
- Store associate and supervisor productivity tools
- Inventory and stockroom process optimization
- Internal dashboards for sales, operations, and training
- AI-powered assistants for customer support and team enablement
- Mobile and web apps for field and retail execution

Given the mix of operational, leadership, and sales-focused roles, we believe there may be strong opportunities to reduce manual effort and improve consistency across sto